# Infra-FM: Multi-Continent Pretraining

Pretrains a SimCLR encoder across all available regional datasets combined.

**Regions included:**
- central-america (~1,782 tiles)
- australia-oceania (~3,200 tiles est.)
- africa (~5,500 tiles est.)
- south-america (~9,000 tiles est.)

**Config (from ablation grid):**
- temperature=0.1 (best in grid)
- projection_dim=128
- epochs=200

**Before running:**
- Runtime → Change runtime type → GPU (T4)
- All regional datasets uploaded to `My Drive/infra_fm/datasets/`
- Code zip at `My Drive/infra_fm/code/infra_fm_curation.zip`

## 1. Mount Drive + check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Go to Runtime -> Change runtime type -> GPU.')

## 2. Install dependencies

In [ ]:
%%capture
!pip install scipy opencv-python-headless

## 3. Extract code

In [ ]:
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infra_fm_clean'
CODE_ROOT  = f'{EXTRACT_TO}/infra_fm_code_only'

if not Path(f'{CODE_ROOT}/downstream').exists():
    print('Extracting code...')
    os.makedirs(EXTRACT_TO, exist_ok=True)
    with zipfile.ZipFile(CODE_ZIP, 'r') as z:
        for member in z.namelist():
            clean_path = member.replace('\\', '/')
            target = os.path.join(EXTRACT_TO, clean_path)
            if clean_path.endswith('/'):
                os.makedirs(target, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target), exist_ok=True)
                with z.open(member) as src, open(target, 'wb') as dst:
                    dst.write(src.read())
    print('Done.')
else:
    print('Already extracted.')

sys.path.insert(0, CODE_ROOT)
os.chdir(CODE_ROOT)

from downstream.common.models import SimCLRModel
from downstream.common.utils import ensure_dir, set_seed
from pretraining.augmentations import TwoCropTransform, build_simclr_transform
from pretraining.losses import nt_xent_loss
from pretraining.datasets import InfrastructureImageDataset
print('Imports OK')

## 4. Configuration

In [ ]:
import json

DATASETS_DIR = f'{DRIVE_ROOT}/datasets'
OUTPUT_DIR   = f'{DRIVE_ROOT}/results/pretrain_global'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Regions to include — add more as they become available
REGIONS = [
    'central-america',
    'australia-oceania',
    'africa',
    'south-america',
    'europe',
    'asia',
    'north-america',
]

# Best config from ablation grid
TEMPERATURE    = 0.1
PROJECTION_DIM = 128
EPOCHS         = 100
BATCH_SIZE     = 256                  # larger batch benefits contrastive learning
LEARNING_RATE  = 1.2e-3               # from SimCLR paper, scaled with batch size
BAND_INDICES   = '0,1,2,3,4,5,6,7,8'  # sentinel2_ms + sentinel1 (9 bands)
N_OPTICAL      = 7
BACKBONE       = 'resnet18'
SAVE_EVERY     = 25
SEED           = 42
IMAGE_SIZE     = 224

# Check which datasets are available
print('Dataset availability:')
available = []
total_tiles = 0
for region in REGIONS:
    dataset_path = Path(f'{DATASETS_DIR}/dataset_{region}_stac_v1')
    manifest_path = dataset_path / 'manifest.json'
    if manifest_path.exists():
        manifest = json.load(open(manifest_path))
        n = manifest['n_tiles']
        mods = manifest.get('modalities', [])
        sample_shape = manifest['records'][0].get('image_shape', [])
        print(f'  {region:25s} {n:>6,} tiles | {mods} | shape={sample_shape}')
        available.append(region)
        total_tiles += n
    else:
        print(f'  {region:25s} NOT FOUND — {dataset_path}')

print(f'\nTotal tiles for pretraining: {total_tiles:,}')
print(f'Config: epochs={EPOCHS}, temperature={TEMPERATURE}, '
      f'projection_dim={PROJECTION_DIM}, batch={BATCH_SIZE}')

# Estimate time
# ~0.05s/tile/epoch on T4 at batch_size=64
est_min = (total_tiles * EPOCHS * 0.05) / 60
print(f'Estimated time on T4: ~{est_min:.0f} minutes ({est_min/60:.1f} hours)')

## 5. Build combined dataset

In [ ]:
import shutil, time
from pathlib import Path

LOCAL_DATASETS = '/content/datasets'
os.makedirs(LOCAL_DATASETS, exist_ok=True)

for region in REGIONS:
    src = Path(f'{DATASETS_DIR}/dataset_{region}_stac_v1')
    dst = Path(f'{LOCAL_DATASETS}/dataset_{region}_stac_v1')
    if not src.exists():
        print(f'  {region:25s} skipped (not in Drive)')
        continue
    if dst.exists():
        print(f'  {region:25s} already in local')
        continue
    t0 = time.time()
    print(f'  {region:25s} copying...', end=' ', flush=True)
    shutil.copytree(src, dst)
    print(f'done in {time.time()-t0:.0f}s')

# Switch DATASETS_DIR for cell 10
DATASETS_DIR = LOCAL_DATASETS
print(f'\nDATASETS_DIR = {DATASETS_DIR}')

In [ ]:
import numpy as np
import torch
from torch.utils.data import ConcatDataset, DataLoader

transform = TwoCropTransform(
    build_simclr_transform(IMAGE_SIZE, n_optical=N_OPTICAL)
)

datasets = []
for region in available:
    dataset_path = f'{DATASETS_DIR}/dataset_{region}_stac_v1'
    try:
        ds = InfrastructureImageDataset(
            dataset_root=dataset_path,
            transform=transform,
            band_indices=BAND_INDICES,
        )
        datasets.append(ds)
        print(f'  {region:25s} {len(ds):>6,} samples loaded')
    except Exception as e:
        print(f'  {region:25s} FAILED: {e}')

combined = ConcatDataset(datasets)
print(f'\nCombined dataset: {len(combined):,} tiles')

loader = DataLoader(
    combined,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    drop_last=True,
)
print(f'Batches per epoch: {len(loader)}')

## 6. Run pretraining

In [ ]:
import time, csv
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

set_seed(SEED)
in_channels = len([int(x) for x in BAND_INDICES.split(',')])

model = SimCLRModel(
    backbone_name=BACKBONE,
    projection_dim=PROJECTION_DIM,
    pretrained_backbone=False,
    in_channels=in_channels,
).to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Cosine annealing — smoothly decays LR to 0 over training
# Better than fixed LR for longer runs
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

scaler = GradScaler(device='cuda', enabled=(device.type == 'cuda'))

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params:,}')
print(f'in_channels: {in_channels}')

# Check for existing checkpoint to resume from
best_pt    = Path(OUTPUT_DIR) / 'best.pt'
resume_pt  = Path(OUTPUT_DIR) / 'latest.pt'
start_epoch = 1
best_loss   = float('inf')

if resume_pt.exists():
    print(f'\nResuming from {resume_pt}...')
    ckpt = torch.load(resume_pt, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_loss   = ckpt.get('best_loss', float('inf'))
    print(f'Resuming from epoch {start_epoch}, best_loss={best_loss:.4f}')
else:
    print('Starting fresh.')

# Save config
config = {
    'regions':        available,
    'total_tiles':    len(combined),
    'epochs':         EPOCHS,
    'temperature':    TEMPERATURE,
    'projection_dim': PROJECTION_DIM,
    'batch_size':     BATCH_SIZE,
    'learning_rate':  LEARNING_RATE,
    'band_indices':   BAND_INDICES,
    'in_channels':    in_channels,
    'backbone':       BACKBONE,
    'scheduler':      'cosine_annealing',
}
with open(f'{OUTPUT_DIR}/train_config.json', 'w') as f:
    json.dump(config, f, indent=2)

# Training loop
loss_log_path = Path(OUTPUT_DIR) / 'loss_log.csv'
log_mode = 'a' if resume_pt.exists() else 'w'

with open(loss_log_path, log_mode, newline='') as log_f:
    writer = csv.DictWriter(log_f, fieldnames=['epoch', 'loss', 'lr', 'elapsed_s'])
    if log_mode == 'w':
        writer.writeheader()

    run_start = time.time()

    for epoch in range(start_epoch, EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        losses = []

        for batch in loader:
            x1, x2 = batch['image']
            x1, x2 = x1.to(device, non_blocking=True), x2.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type=device.type,
                          enabled=(device.type == 'cuda')):
                _, z1 = model(x1)
                _, z2 = model(x2)
                loss = nt_xent_loss(z1, z2, temperature=TEMPERATURE)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())

        scheduler.step()
        epoch_loss = sum(losses) / max(len(losses), 1)
        current_lr = scheduler.get_last_lr()[0]
        elapsed    = time.time() - epoch_start

        writer.writerow({
            'epoch': epoch, 'loss': round(epoch_loss, 6),
            'lr': round(current_lr, 8), 'elapsed_s': round(elapsed, 1)
        })
        log_f.flush()

        # Print every 10 epochs
        if epoch % 10 == 0 or epoch == EPOCHS or epoch == 1:
            total_elapsed = time.time() - run_start
            remaining = (total_elapsed / (epoch - start_epoch + 1)) * (EPOCHS - epoch)
            print(f'Epoch {epoch:03d}/{EPOCHS} | loss={epoch_loss:.4f} | '
                  f'lr={current_lr:.2e} | {elapsed:.1f}s/ep | '
                  f'~{remaining/60:.0f}min remaining')

        # Save best
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss': best_loss,
                'best_loss': best_loss,
                'config': config,
                'band_indices': BAND_INDICES,
                'in_channels': in_channels,
            }, best_pt)

        # Save latest (for resuming)
        if epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss': epoch_loss,
                'best_loss': best_loss,
                'config': config,
                'band_indices': BAND_INDICES,
                'in_channels': in_channels,
            }, resume_pt)
            # Also save epoch snapshot
            epoch_pt = Path(OUTPUT_DIR) / f'epoch_{epoch:03d}.pt'
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'loss': epoch_loss, 'config': config,
                        'band_indices': BAND_INDICES, 'in_channels': in_channels},
                       epoch_pt)

total_time = time.time() - run_start
print(f'\nPretraining complete in {total_time/60:.1f} minutes')
print(f'Best loss: {best_loss:.4f}')
print(f'Checkpoint: {best_pt}')

In [ ]:

import inspect
from downstream.common.models import SimCLRModel
src = inspect.getsource(SimCLRModel.__init__)
if 'net.fc = nn.Identity()' in src and 'self.backbone = net' in src:
    print('GOOD: SimCLRModel has the correct architecture')
    print('  net.fc = nn.Identity() ✓')
    print('  self.backbone = net ✓')
else:
    print('BAD: SimCLRModel still has old architecture — fix before rezipping')
print()
print(src[:500])

## 7. Plot loss curve

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv(f'{OUTPUT_DIR}/loss_log.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(log['epoch'], log['loss'], color='steelblue', linewidth=1.5)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('NT-Xent Loss')
ax1.set_title('Pretraining Loss')
ax1.grid(alpha=0.3)

ax2.plot(log['epoch'], log['lr'], color='orange', linewidth=1.5)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning Rate')
ax2.set_title('Cosine LR Schedule')
ax2.grid(alpha=0.3)

plt.suptitle(f'Multi-continent pretraining | '
             f'{len(combined):,} tiles | temp={TEMPERATURE} | '
             f'pd={PROJECTION_DIM}', fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {OUTPUT_DIR}/loss_curve.png')

## 8. Quick sanity check — verify checkpoint loads correctly

In [ ]:
ckpt = torch.load(f'{OUTPUT_DIR}/best.pt', map_location='cpu')
print('Checkpoint contents:')
print(f'  Epoch:       {ckpt["epoch"]}')
print(f'  Best loss:   {ckpt["loss"]:.4f}')
print(f'  in_channels: {ckpt["in_channels"]}')
print(f'  band_indices:{ckpt["band_indices"]}')
print(f'  Keys:        {len(ckpt["model_state_dict"])}')

# Verify it loads into SimCLRModel cleanly
test_model = SimCLRModel(
    backbone_name='resnet18',
    projection_dim=PROJECTION_DIM,
    pretrained_backbone=False,
    in_channels=ckpt['in_channels'],
)
missing, unexpected = test_model.load_state_dict(
    ckpt['model_state_dict'], strict=False
)
print(f'  Missing keys:    {len(missing)}')
print(f'  Unexpected keys: {len(unexpected)}')
if len(missing) == 0:
    print('  Checkpoint loads perfectly.')
else:
    print(f'  WARNING: {len(missing)} missing keys — check architecture')